<div style="background: linear-gradient(135deg, #0078d4 0%, #106ebe 50%, #005a9e 100%); color: white; padding: 30px; border-radius: 12px; margin: 20px 0; box-shadow: 0 4px 15px rgba(0, 120, 212, 0.3);">
    <h1 style="margin: 0; text-align: center; font-size: 2.2em; font-weight: 600; letter-spacing: 0.5px; font-family: 'Segoe UI', -apple-system, BlinkMacSystemFont, Roboto, 'Helvetica Neue', sans-serif;">
        Build 2026 Demo, Qwen 32B SFT
    </h1>
</div>

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 10px; margin: 20px 0;">
    <h3 style="margin: 0; text-align: center;">Sections Breakdown</h3>
</div>

<ol style="color: #2c3e50; line-height: 1.8;">
<li>🛠️ <b>Setup:</b> Configure paths and select the Foundry project</li>
<li>🔍 <b>Confirm Submit Parameters:</b> Inspect cluster config, datasets, and training knobs</li>
<li>🚀 <b>Submit the Job:</b> Assemble the job spec and submit via the <code>azure-ai-projects</code> SDK</li>
<li>📈 <b>Tail Rollouts:</b> Pull artifacts and write <code>grades.csv</code>, <code>train_curve.csv</code>, and <code>eval_curve.csv</code></li>
</ol>

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 10px; margin: 20px 0;">
    <h3 style="margin: 0; text-align: center;">What This Run Does</h3>
</div>

<ul style="color: #2c3e50; line-height: 1.8;">
<li><b>Model:</b> Qwen/Qwen3-32B (HF-downloaded inside the container)</li>
<li><b>Approach:</b> Async GRPO fine-tuning with env reward (Taubench retail agent / Retail domain)</li>
<li><b>Entrypoint:</b> <code>retail_slime_train.py</code> via SLIME on Ray</li>
<li><b>Cluster choice:</b> <code>"h100"</code> (ND96r_H100_v5) or <code>"a100"</code> (ND96amsr_A100_v4) &mdash; pass to <code>submit_job(cluster=...)</code></li>
<li><b>Layout:</b> 4 nodes &times; 8 GPUs &mdash; 2 actor (TP=8) + 2 rollout (TP=8 SGLang)</li>
<li><b>Datasets:</b> <code>retail-train-data</code>, <code>retail-code</code>, plus an SFT LoRa seed checkpoint</li>
<li><b>Outputs:</b> <code>model_output</code>, <code>checkpoints</code>, <code>rollouts</code>, <code>ray_temp</code>, <code>hf_cache</code></li>
</ul>

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 10px; margin: 20px 0;">
    <h3 style="margin: 0; text-align: center;">Prerequisites &amp; Installation</h3>
</div>

In [ ]:
# One-time setup: install the SDKs this notebook needs.
# Re-running is safe (pip will no-op if already installed).
%pip install --quiet --pre -r requirements.txt \
    --extra-index-url https://pkgs.dev.azure.com/azure-sdk/public/_packaging/azure-sdk-for-python/pypi/simple
print('Setup complete. Required packages installed.')
print('If this is the first time, also run `az login` in a terminal,')
print('then restart the kernel before continuing.')

<div style="background: #e7f3ff; border: 1px solid #b3d9ff; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <p style="margin: 0; color: #0066cc;">
        <strong>💡 Note:</strong> Make sure your Foundry CLI is authenticated (<code>az login</code>) to the tenant that owns your project before running the submission cell.
    </p>
</div>

<div style="background: linear-gradient(135deg, #0078d4 0%, #106ebe 50%, #005a9e 100%); color: white; padding: 30px; border-radius: 12px; margin: 20px 0; box-shadow: 0 4px 15px rgba(0, 120, 212, 0.3);">
    <h1 style="margin: 0; text-align: center; font-size: 2.2em; font-weight: 600; letter-spacing: 0.5px; font-family: 'Segoe UI', -apple-system, BlinkMacSystemFont, Roboto, 'Helvetica Neue', sans-serif;">
        Recipe Setup
    </h1>
</div>

## <span style="font-size:0.8em;"> </span>

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 10px; margin: 20px 0;">
    <h3 style="margin: 0; text-align: center;">🛠️ Section 1. Paths and Environment</h3>
</div>

<p>Discovers the recipe and reports directories relative to this notebook so the bundle is self-contained, and selects the Foundry project to submit against.</p>

In [ ]:
from slime_sft_setup import setup_env

# Pass the Foundry project you have access to plus the Azure workspace
# coordinates that own it. Region is the workspace region (e.g.
# westcentralus, eastus2). Workspace follows the format
# <workspace>@<project>@AML.
setup_env(
    project="<your-foundry-project-name>",
    subscription="<your-subscription-id>",
    resource_group="<your-resource-group>",
    workspace="<your-workspace>@<your-foundry-project-name>@AML",
    region="<your-region>",
)

<div style="background: linear-gradient(135deg, #0078d4 0%, #106ebe 50%, #005a9e 100%); color: white; padding: 30px; border-radius: 12px; margin: 20px 0; box-shadow: 0 4px 15px rgba(0, 120, 212, 0.3);">
    <h1 style="margin: 0; text-align: center; font-size: 2.2em; font-weight: 600; letter-spacing: 0.5px; font-family: 'Segoe UI', -apple-system, BlinkMacSystemFont, Roboto, 'Helvetica Neue', sans-serif;">
        Job Submission &amp; Monitoring
    </h1>
</div>

## <span style="font-size:0.8em;"> </span>

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 10px; margin: 20px 0;">
    <h3 style="margin: 0; text-align: center;">🔍 Section 2. Confirm Cluster, Datasets, and Training Knobs</h3>
</div>

<p>Echoes the chosen cluster config (name, instance type, name prefix), the Foundry dataset URIs (<code>train_dataset</code>, <code>code_dataset</code>, <code>sft_lora_dataset</code>), the container image, and a handful of key training knobs from <code>submit_sft.py</code>. Run this <em>before</em> submission to verify the recipe.</p>

In [ ]:
from slime_sft_setup import show_submit_params

# Choose "h100" or "a100".
show_submit_params(cluster="h100")

## <span style="font-size:0.8em;"> </span>

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 10px; margin: 20px 0;">
    <h3 style="margin: 0; text-align: center;">🚀 Section 3. Submit the Job</h3>
</div>

<p>Assembles a <code>CommandJob</code> from the recipe's payload builder and submits it via the <code>azure-ai-projects</code> SDK. The returned <code>JOB_ID</code> is stored on <code>ENV</code> for the rollout-tailing cell below.</p>
<p>Pass <code>cluster=&quot;a100&quot;</code> to target the A100 cluster, or override <code>instance_count=</code> / <code>name=</code> as needed.</p>

<div style="background: #e7f3ff; border: 1px solid #b3d9ff; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <p style="margin: 0; color: #0066cc;">
        <strong>💡 Note:</strong> This step actually launches the training run on the target cluster. Re-running it will create a new job with a new suffix &mdash; previously submitted jobs are not affected.
    </p>
</div>

In [ ]:
from slime_sft_setup import submit_job

submit_job(cluster="h100")

## <span style="font-size:0.8em;"> </span>

<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 20px; border-radius: 10px; margin: 20px 0;">
    <h3 style="margin: 0; text-align: center;">📈 Section 4. Tail Rollouts (run after the job starts producing data)</h3>
</div>

<p>Once the job has started writing artifacts, this cell pulls them down and writes three CSVs into <code>reports/out_&lt;JOB_ID&gt;/</code>:</p>
<ul>
<li><code>grades.csv</code> &mdash; per-rollout grader scores</li>
<li><code>train_curve.csv</code> &mdash; training reward by step</li>
<li><code>eval_curve.csv</code> &mdash; held-out val reward by checkpoint</li>
</ul>

In [ ]:
from slime_sft_setup import tail_rollouts
tail_rollouts()